# Lezione 29 — Contraddizioni: quando una memoria nuova smentisce una vecchia

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 28
(retrieval ibrido). Chiude la **Fase 4**. Finora ogni memoria e' stata
trattata come sempre valida; oggi affrontiamo il caso in cui non lo e'
piu': una preferenza cambia, e l'archivio deve accorgersene senza
cancellare la storia.

**Cosa saprai fare alla fine:** rilevare una contraddizione tra due
memorie `preference` con un'euristica dichiarata (polarita' opposta +
argomento condiviso), applicare una politica di aggiornamento che non
cancella mai (segna come superata, mantiene la storia), e riconoscere
onestamente i limiti dell'euristica su casi che non copre.

## Parte 1 — Teoria: contraddizione, non solo eta'

La Lezione 23 aveva gia' anticipato il problema: una `preference` non
decade con il tempo (a differenza di `episodic`), ma puo' essere
**sostituita** da un'espressione piu' recente della stessa preferenza. Il
tempo da solo (Lezione 24) non basta a distinguere "questa preferenza e'
ancora valida" da "questa preferenza e' stata smentita": serve confrontare
il **contenuto** di due memorie, non solo la loro data.

Senza un modello linguistico (nessuna credenziale, nessuna GPU in questo
ambiente — lo stesso limite gia' dichiarato per il NER della Lezione 26),
costruiamo un'euristica su due segnali osservabili nel testo:

- **polarita'** — `positiva` se il testo contiene `likes`/`prefers`,
  `negativa` se contiene `dislikes`; nessuna delle due altrimenti;
- **argomento** — le parole di contenuto del testo, dopo aver tolto il
  prefisso fisso (`"The user prefers/likes/dislikes "`) e le parole
  vuote (`the`, `for`, `about`, ...).

Due memorie `preference` sono in **conflitto** se hanno polarita'
**opposta** e condividono almeno una parola di argomento. `"The user
likes walking meetings."` e `"The user dislikes walking meetings."`
condividono `{walking, meetings}` con polarita' opposta: conflitto. `"The
user prefers morning sessions..."` e `"The user prefers short
updates..."` sulla stessa cosa hanno **la stessa** polarita'
(`positiva` entrambe): nessun conflitto secondo questa euristica, anche
se descrivono preferenze diverse — un limite dichiarato, non nascosto,
che vedrai concretamente nella Parte 3.

**La politica di aggiornamento**: quando arriva una memoria che
contraddice una memoria esistente, **non si cancella** la vecchia — si
marca come `superseded_by` (sostituita da) il `memory_id` della nuova.
Stesso principio dei flag di audit della Lezione 1: perdere
silenziosamente informazione (qui: che una preferenza *era* vera fino a un
certo punto) e' peggio di tenerla, marcata come non piu' attuale.

In [1]:
import re

STOPWORD = {'the', 'user', 'a', 'an', 'for', 'about', 'of', 'to', 'il', 'la', 'le', 'i'}


def polarita(testo):
    t = testo.lower()
    if 'dislikes' in t:
        return 'negativa'
    if 'likes' in t or 'prefers' in t:
        return 'positiva'
    return None


def argomento(testo):
    t = re.sub(r'^\s*the user (prefers|likes|dislikes)\s+', '', testo.lower())
    t = t.rstrip('.')
    return {p for p in re.findall(r'[a-z\u00e0-\u00fc]+', t) if p not in STOPWORD}


def in_conflitto(testo_a, testo_b):
    pol_a, pol_b = polarita(testo_a), polarita(testo_b)
    if pol_a is None or pol_b is None or pol_a == pol_b:
        return False
    return len(argomento(testo_a) & argomento(testo_b)) > 0


a = 'The user likes walking meetings.'
b = 'The user dislikes walking meetings.'
c = 'The user prefers morning sessions for il colloquio.'
print(f'a vs b (polarita\' opposta, stesso argomento): {in_conflitto(a, b)}')
print(f'a vs c (argomenti diversi)                  : {in_conflitto(a, c)}')

a vs b (polarita' opposta, stesso argomento): True
a vs c (argomenti diversi)                  : False


Leggi l'output: `a` e `b` sono in conflitto (polarita' opposta,
`{walking, meetings}` condiviso); `a` e `c` no (nessuna parola di
argomento in comune, oltre alla polarita' diversa dal tipo di frase).

## Parte 2 — Esercizio guidato

Il tuo compito: verifica che `"The user dislikes late notifications."` e
`"The user likes late notifications."` siano in conflitto secondo
`in_conflitto`, e che invece `"The user dislikes late notifications."` e
`"The user dislikes loud notifications."` **non** lo siano (stessa
polarita', nessun conflitto per costruzione dell'euristica).

In [2]:
# Scrivi qui: chiama in_conflitto sulle due coppie indicate e stampa i risultati.

pass

### Soluzione spiegata

La prima coppia ha polarita' opposta (`negativa` vs `positiva`) e
condivide `{late, notifications}`: conflitto. La seconda coppia ha la
**stessa** polarita' (`negativa` per entrambe): l'euristica restituisce
sempre `False` quando le polarita' coincidono, indipendentemente
dall'argomento — anche se nella realta' "dislikes late notifications" e
"dislikes loud notifications" potrebbero coesistere senza problemi (non
sono nemmeno una vera contraddizione da rilevare).

In [3]:
caso_1 = in_conflitto('The user dislikes late notifications.', 'The user likes late notifications.')
caso_2 = in_conflitto('The user dislikes late notifications.', 'The user dislikes loud notifications.')
print(f'caso 1 (polarita\' opposta, stesso argomento): {caso_1}')
print(f'caso 2 (stessa polarita\'):                    {caso_2}')
assert caso_1 is True
assert caso_2 is False

caso 1 (polarita' opposta, stesso argomento): True
caso 2 (stessa polarita'):                    False


## Parte 3 — Il progetto: Memory AI Lab, passo 29 — arriva un nuovo batch, si aggiorna l'archivio

Come nelle Lezioni 1-2 (un nuovo batch di memorie che arriva nel tempo),
simuliamo tre nuove memorie `preference` in arrivo e le confrontiamo con
l'archivio storico di `preference` del train set. Per ognuna: se e' in
conflitto con memorie esistenti, quelle vecchie vengono marcate
`superseded_by` (mai cancellate).

In [4]:
import pandas as pd
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
train = pd.read_csv(processed / 'memory_train.csv')
archivio_preference = train[train['type'] == 'preference'].reset_index(drop=True)
archivio_preference['superseded_by'] = None

nuove_memorie = pd.DataFrame([
    {'memory_id': 'nuova_001', 'text': 'The user dislikes walking meetings.'},
    {'memory_id': 'nuova_002', 'text': 'The user likes late notifications.'},
    {'memory_id': 'nuova_003', 'text': 'The user prefers evening sessions for il progetto TensorFlow.'},
])

for _, nuova in nuove_memorie.iterrows():
    # apply() sull'intero archivio (non su un sottoinsieme filtrato): la maschera
    # booleana risultante ha sempre lo stesso indice di archivio_preference,
    # cosi' .loc[maschera, ...] resta allineato a ogni iterazione del ciclo.
    in_conflitto_con = archivio_preference.apply(
        lambda riga: riga['superseded_by'] is None and in_conflitto(nuova['text'], riga['text']),
        axis=1)
    n_conflitti = in_conflitto_con.sum()
    archivio_preference.loc[in_conflitto_con, 'superseded_by'] = nuova['memory_id']
    print(f"\"{nuova['text']}\" -> {n_conflitti} memorie storiche marcate superseded")

"The user dislikes walking meetings." -> 13 memorie storiche marcate superseded
"The user likes late notifications." -> 17 memorie storiche marcate superseded
"The user prefers evening sessions for il progetto TensorFlow." -> 0 memorie storiche marcate superseded


In [5]:
superate = archivio_preference['superseded_by'].notna().sum()
ancora_valide = archivio_preference['superseded_by'].isna().sum()
print(f'memorie preference storiche superate : {superate}')
print(f'memorie preference storiche ancora valide: {ancora_valide}')
print(f'totale (nessuna cancellata)               : {len(archivio_preference)}')
assert superate + ancora_valide == len(archivio_preference)  # nessuna riga persa, solo marcata

esempio_superata = archivio_preference[archivio_preference['superseded_by'].notna()].iloc[0]
print(f"\nesempio: {esempio_superata['memory_id']} (\"{esempio_superata['text']}\") "
      f"superseded_by={esempio_superata['superseded_by']}")

memorie preference storiche superate : 30
memorie preference storiche ancora valide: 31
totale (nessuna cancellata)               : 61

esempio: hist_0000 ("The user likes walking meetings.") superseded_by=nuova_001


Guarda il terzo caso onestamente: `"The user prefers evening sessions for
il progetto TensorFlow."` **non** ha marcato nessuna memoria storica come
superata, nonostante descriva una preferenza di orario diversa
("evening" contro "morning" della memoria storica sullo stesso progetto)
— l'euristica di questa lezione rileva solo conflitti di polarita'
(likes/dislikes), non conflitti tra valori diversi della stessa
dimensione (mattina/sera). E' un limite reale, non un bug: un vero
sistema di gestione delle contraddizioni avrebbe bisogno di riconoscere
categorie di preferenza (orario, formato, canale) e valori mutuamente
esclusivi all'interno di ciascuna — un problema piu' complesso, fuori
scope per un'euristica basata su parole chiave.

## Cosa hai imparato — e la Fase 4 e' completa

- Una **contraddizione** tra memorie richiede di confrontare il
  **contenuto**, non solo il tempo: la Lezione 24 (recency) non basta da
  sola a gestire preferenze che cambiano.
- Un'euristica dichiarata (polarita' opposta + argomento condiviso) rileva
  **alcuni** conflitti reali, ma ha limiti espliciti: non riconosce
  contraddizioni tra valori diversi della stessa dimensione senza una
  polarita' esplicita likes/dislikes.
- La politica di aggiornamento corretta **non cancella mai**: marca la
  memoria superata (`superseded_by`), mantenendo la storia — lo stesso
  principio dei flag di audit fin dalla Lezione 1.
- **Chiudi qui la Fase 4.** Dallo schema esplicito (Lezione 22) al
  retrieval ibrido con gestione delle contraddizioni (oggi), il progetto
  ora rappresenta le memorie non solo come testo classificato, ma come
  record con tempo, importanza, relazioni ed evoluzione nel tempo.

## Quiz

1. Perche' il tempo da solo (recency, Lezione 24) non basta a decidere se
   una preferenza e' ancora valida?
2. Perche' la politica di aggiornamento marca le memorie superate invece
   di cancellarle?
3. L'euristica non rileva il conflitto tra "morning sessions" e "evening
   sessions" per lo stesso progetto. Perche', e come lo spiegheresti a
   qualcuno che si aspettava che lo rilevasse?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' una preferenza non decade con il passare del tempo (Lezione
   23): resta valida finche' non viene esplicitamente contraddetta da una
   nuova affermazione, non per una semplice scadenza temporale. Serve
   confrontare *cosa dice* la nuova memoria rispetto a quelle esistenti,
   non solo *quando* e' stata registrata.
2. Perche' cancellare perderebbe per sempre l'informazione che quella
   preferenza *era* vera fino a un certo punto — utile per capire
   l'evoluzione dell'utente nel tempo, non solo il suo stato attuale.
   Marcare `superseded_by` mantiene la storia completa, esattamente come
   i flag `_was_missing` della Lezione 1 mantengono traccia delle
   imputazioni invece di nasconderle.
3. Perche' l'euristica rileva conflitti solo tramite le parole chiave
   `likes`/`prefers`/`dislikes` (polarita' esplicita): "morning" ed
   "evening" non sono polarita' opposte secondo questa definizione, sono
   due valori diversi della stessa dimensione (l'orario). Un'euristica
   piu' sofisticata (o un modello linguistico) sarebbe necessaria per
   riconoscere anche questo tipo di conflitto — dichiarare il limite,
   come fa questa lezione, e' meglio che promettere una copertura che il
   codice non ha.
</details>

## Fonti

- Python, documentazione ufficiale, modulo `re` (gia' citato nella
  Lezione 26, riusato qui per l'estrazione dell'argomento):
  https://docs.python.org/3/library/re.html

## Prossimo modulo

La Fase 4 (rappresentare le memorie) e' chiusa: schema, type, recency,
importanza, entita', grafo, retrieval ibrido, contraddizioni. Il modulo
successivo affronta i **Transformer**: l'attention, il meccanismo che
rende gli LLM capaci di generare testo — la base di Gemma, il modello
open che il corso user'a nella Fase 6.